# Lab 1.5 · Causal attention

**15–20 minutes · CPU**

Read the [lab guide](README.md), then run the setup. Replace each `return None` with your implementation and rerun its check.

In [ ]:
from math import exp, isclose, sqrt

queries = [[1, 0], [1, 0], [0, 1]]
keys = [[1, 0], [0, 1], [2, 1]]
values = [[1, 0], [0, 2], [3, 3]]

def attention_weights(scores):
    shifted = [exp(score - max(scores)) for score in scores]
    total = sum(shifted)
    return [weight / total for weight in shifted]

## 1. Score and mask keys

For query index i and key index j, compute their dot product divided by the square root of the key width. If causal masking is on and j is greater than i, use negative infinity.

In [ ]:
def score_row(i, queries, keys, causal):
    return None

In [ ]:
scores = score_row(0, queries, keys, True)
if scores is not None:
    assert isclose(scores[0], 1 / sqrt(2))
    assert scores[1:] == [float("-inf")] * 2
    assert all(score != float("-inf") for score in score_row(0, queries, keys, False))
    print("PASS — first weight row:", attention_weights(scores))

## 2. Mix the values

Compute a weighted sum for each value feature. Then compare the output with the causal mask on and off. Predict which rows change when only the final value vector changes.

In [ ]:
def mix_values(weights, values):
    return None

In [ ]:
if scores is not None and mix_values([1, 0, 0], values) is not None:
    def attention(values, causal):
        weights = [attention_weights(score_row(i, queries, keys, causal))
                   for i in range(len(queries))]
        assert all(isclose(sum(row), 1.0) for row in weights)
        return weights, [mix_values(row, values) for row in weights]

    masked_weights, masked = attention(values, True)
    _, unmasked = attention(values, False)
    changed = [*values[:-1], [30, 30]]
    _, masked_changed = attention(changed, True)
    _, unmasked_changed = attention(changed, False)
    assert masked[0] == values[0]
    assert masked_changed[:2] == masked[:2] and masked_changed[2] != masked[2]
    assert unmasked_changed[0] != unmasked[0]
    pairs = [(i, j) for i, row in enumerate(masked_weights) for j, w in enumerate(row) if w > 0]
    print("Visible query-key pairs:", len(pairs), "of", len(queries) * len(keys))
    for row in masked_weights:
        print([round(w, 3) for w in row])
    print("PASS — early causal outputs ignore future values")

## Wrap-up

In a new cell, append a fourth query and key. Use `score_row` and `attention_weights` to count the visible pairs again. Predict the new count before running that cell. Does a causal mask alone save dot products in every attention implementation?